In [ ]:
from PIL import Image

In [ ]:

def hide_message(image_path, message):
    img = Image.open(image_path)
    # Check if the image is black and white
    if img.mode != "L":
        # Convert the image to black and white
        img = img.convert("L")

    binary_message = ''.join(format(ord(i), '08b') for i in message)

    width, height = img.size

    max_message_length = (width * height) // 8

    if len(binary_message) > max_message_length:
        raise ValueError('Message too long to be hidden in the given image')

    new_img = Image.new(img.mode, img.size)

    message_index = 0

    # Loop through each pixel in the image - use nested loop where x represents width & y represents height
            # Get the RGB values of the current pixel, using img.getpixel()
            # unpack the result tuple to red, green & blue
    for x in range(width):
       for y in range(height):
            pixel = img.getpixel((x, y))
             # Since it's grayscale, pixel value is a single integer
            gray = pixel

            # Check if the message has been fully hidden
            if message_index == len(binary_message):
                new_img.putpixel((x, y), gray)  # Use gray value for grayscale image
                continue

            # Modify the least significant bit of the gray value with the message bits
            gray_binary = format(gray, '08b')

            #update last value of gray_binary and increment message_index
            gray_binary = gray_binary[:-1] + binary_message[message_index]
            message_index += 1

            #converting back to binary
            gray = int(gray_binary, 2)

            # Create a new pixel with the modified gray value and add it to the new image
            new_img.putpixel((x, y), gray) # Use gray value for grayscale image


    # Save the new image with the message hidden inside
    new_img.save('hidden.png')

In [ ]:
hide_message("/content/kerberos.jpg", "I've got massive text size when in a message... the font in list view of messages is normal and in other apps ")

In [ ]:
from PIL import Image

def extract_message(image_path):
    img = Image.open(image_path)
    width, height = img.size
    binary_message = ''
    message_extracted = False  # Flag to indicate if message has been extracted

    for x in range(width):
        for y in range(height):
            if message_extracted:  # Stop if message has already been extracted
                break

            gray = img.getpixel((x, y))
            gray_bit = format(gray, '08b')[-1]  # Get the least significant bit (LSB)
            binary_message += gray_bit

            # Check if we've read enough bits to form a byte
            if len(binary_message) % 8 == 0:
                byte = binary_message[-8:]  # Get the last 8 bits

                # Check for null terminator
                if byte == '00000000':  # Null terminator found
                    message_extracted = True  # Set flag to indicate message extraction
                    break  # Exit inner loop
                else:
                    # Check if the byte represents a printable ASCII character
                    char_value = int(byte, 2)
                    if not (32 <= char_value <= 126):  # Non-printable character
                        message_extracted = True  # Stop extracting at non-printable char
                        break  # Exit inner loop

        if message_extracted:  # Stop if message has already been extracted
            break  # Exit outer loop

    # Now create the message from the extracted binary bits
    message = ''
    for i in range(0, len(binary_message) - 8, 8):  # Skip the last 8 bits (null terminator)
        byte = binary_message[i:i + 8]
        char_value = int(byte, 2)
        message += chr(char_value)

    print(message)


In [ ]:
extract_message("/content/hidden.png")


I've got massive text size when in a message... the font in list view of messages is normal and in other apps 
